In [1]:
import polars as pl
from pathlib import Path
import numpy as np
from scipy.stats import spearmanr, kendalltau

def spearman_ci(x, y, n_boot, rng):
    """Spearman rho with a percentile bootstrap 95% CI."""
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    ok = np.isfinite(x) & np.isfinite(y)
    x, y = x[ok], y[ok]
    n = len(x)
 
    rho0, _ = spearmanr(x, y)
 
    boot_rhos = np.empty(n_boot)
    idx = np.arange(n)
    for b in range(n_boot):
        samp = rng.choice(idx, size=n, replace=True)
        r, _ = spearmanr(x[samp], y[samp])
        boot_rhos[b] = r
 
    lo, hi = np.percentile(boot_rhos, [2.5, 97.5])
    return rho0, lo, hi, n

def kendall_ci(x, y, n_boot, rng):
    """Kendall tau-b with a percentile bootstrap 95% CI."""
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    ok = np.isfinite(x) & np.isfinite(y)
    x, y = x[ok], y[ok]
    n = len(x)
 
    tau0, _ = kendalltau(x, y)
 
    boot_taus = np.empty(n_boot)
    idx = np.arange(n)
    for b in range(n_boot):
        samp = rng.choice(idx, size=n, replace=True)
        t, _ = kendalltau(x[samp], y[samp])
        boot_taus[b] = t
 
    lo, hi = np.percentile(boot_taus, [2.5, 97.5])
    return tau0, lo, hi, n

In [2]:
# Assuming either all *_compressed.tar.zst files or all_groups_lean.tar.zst file is/are decompressed
CWD = Path().resolve()
BASE = CWD.parent
 
print(f"Current working directory (where figure will be saved): {CWD}")
print(f"Folder where groups (data) folder should be: {BASE}")

Current working directory (where figure will be saved): /data/users/bdupin/datasets-organelle-igr/code
Folder where groups (data) folder should be: /data/users/bdupin/datasets-organelle-igr


In [3]:
groups = [
    "fungi_mit",
    "metazoans_mit",
    "plants_mit",
    "plants_plt",
    "protists_mit",
    "protists_plt",
    "green_algae_mit",
    "green_algae_plt",
]
 
polarity_2bin = {
    "++": "same",
    "--": "same",
    "+-": "opp",
    "-+": "opp",
}
 
# --- constants ---
MIN_PAIRS = 5          # per-genome floor on total (same + opposite) pairs; sweep 0/5/10 as sensitivity
N_BOOT = 2000           # bootstrap resamples per (group, target) coefficient
NEGLIGIBLE_BAND = 0.2   # pre-specified |rho| threshold for "no substantial correlation"
SEED = 42
rng = np.random.default_rng(SEED)

In [4]:
rows_s3 = []
 
for g in groups:
    tsv = pl.read_csv(BASE / g / f"{g}.tsv", separator="\t")
    igs = pl.read_csv(BASE / g / f"{g}_summary_igr.tsv", separator="\t")
 
    igs = igs.join(
        tsv.select(["AN", "Genome_length"]),
        on="AN",
        how="left",
        validate="m:1",
    )
 
    igs = igs.with_columns(
        pl.col("Polarity").replace_strict(polarity_2bin).alias("polarity_2bin")
    )
 
    per_genome = igs.group_by("AN").agg(
        pl.col("Genome_length").first().alias("genome_size_bp"),
        pl.len().alias("n_pairs"),
        (pl.col("polarity_2bin") == "opp").sum().alias("n_opposite"),
        pl.col("Length").sum().alias("noncoding_bp"),
    ).sort("AN")
 
    per_genome = per_genome.with_columns(
        (pl.col("n_opposite") / pl.col("n_pairs")).alias("prevalence"),
        (pl.col("noncoding_bp") / pl.col("genome_size_bp")).alias("noncoding_fraction"),
    ).filter(pl.col("n_pairs") >= MIN_PAIRS)
 
    # Spearman's rho with percentile bootstrap 95% CI
    rho_size, lo_size, hi_size, n_size = spearman_ci(
        per_genome["prevalence"], per_genome["genome_size_bp"],
        n_boot=N_BOOT,
        rng=rng
    )
    rho_ncf, lo_ncf, hi_ncf, n_ncf = spearman_ci(
        per_genome["prevalence"], per_genome["noncoding_fraction"],
        n_boot=N_BOOT,
        rng=rng
    )

    # Kendall's tau-b cross-check
    tau_size, tlo_size, thi_size, _ = kendall_ci(
        per_genome["prevalence"], per_genome["genome_size_bp"],
        n_boot=N_BOOT,
        rng=rng
    )
    tau_ncf, tlo_ncf, thi_ncf, _ = kendall_ci(
        per_genome["prevalence"], per_genome["noncoding_fraction"],
        n_boot=N_BOOT,
        rng=rng
    )
 
    rows_s3.append({
        "group": g,
        "N_genomes": per_genome.height,
        "rho_size": rho_size,
        "lo_size": lo_size,
        "hi_size": hi_size,
        "tau_size": tau_size,
        "tlo_size": tlo_size,
        "thi_size": thi_size,
        "rho_ncf": rho_ncf,
        "lo_ncf": lo_ncf,
        "hi_ncf": hi_ncf,
        "tau_ncf": tau_ncf,
        "tlo_ncf": tlo_ncf,
        "thi_ncf": thi_ncf
    })
 
    print(f"{g} (N genomes = {per_genome.height}, min_pairs >= {MIN_PAIRS})")
    print(f"  rho (prevalence, genome size)        = {rho_size:.3f} ({lo_size:.3f},{hi_size:.3f})")
    print(f"  tau (prevalence, genome size)        = {tau_size:.3f} ({tlo_size:.3f},{thi_size:.3f})")
    print(f"  rho (prevalence, noncoding fraction) = {rho_ncf:.3f} ({lo_ncf:.3f},{hi_ncf:.3f})")
    print(f"  tau (prevalence, noncoding fraction) = {tau_ncf:.3f} ({tlo_ncf:.3f},{thi_ncf:.3f})\n")
 
s3 = pl.DataFrame(rows_s3)

fungi_mit (N genomes = 418, min_pairs >= 5)
  rho (prevalence, genome size)        = 0.031 (-0.058,0.126)
  tau (prevalence, genome size)        = 0.018 (-0.044,0.080)
  rho (prevalence, noncoding fraction) = 0.163 (0.069,0.253)
  tau (prevalence, noncoding fraction) = 0.106 (0.045,0.162)

metazoans_mit (N genomes = 16871, min_pairs >= 5)
  rho (prevalence, genome size)        = 0.014 (-0.002,0.031)
  tau (prevalence, genome size)        = 0.008 (-0.003,0.020)
  rho (prevalence, noncoding fraction) = 0.017 (0.001,0.033)
  tau (prevalence, noncoding fraction) = 0.012 (0.001,0.024)

plants_mit (N genomes = 897, min_pairs >= 5)
  rho (prevalence, genome size)        = 0.116 (0.047,0.184)
  tau (prevalence, genome size)        = 0.080 (0.034,0.126)
  rho (prevalence, noncoding fraction) = 0.357 (0.301,0.415)
  tau (prevalence, noncoding fraction) = 0.241 (0.196,0.281)

plants_plt (N genomes = 17034, min_pairs >= 5)
  rho (prevalence, genome size)        = -0.096 (-0.112,-0.079)
  tau (prev

In [6]:
s3_fmt = s3.select(
    pl.col("group"),
    pl.col("N_genomes"),
    pl.format(
        "{} ({}, {})",
        pl.col("rho_size").round(3),
        pl.col("lo_size").round(3),
        pl.col("hi_size").round(3),
    ).alias("Spearman rho (prevalence vs genome size)"),
    pl.format(
        "{} ({}, {})",
        pl.col("tau_size").round(3),
        pl.col("tlo_size").round(3),
        pl.col("thi_size").round(3),
    ).alias("Kendall tau-b (prevalence vs genome size)"),
    pl.format(
        "{} ({}, {})",
        pl.col("rho_ncf").round(3),
        pl.col("lo_ncf").round(3),
        pl.col("hi_ncf").round(3),
    ).alias("Spearman rho (prevalence vs noncoding fraction)"),
    pl.format(
        "{} ({}, {})",
        pl.col("tau_ncf").round(3),
        pl.col("tlo_ncf").round(3),
        pl.col("thi_ncf").round(3),
    ).alias("Kendall tau-b (prevalence vs noncoding fraction)"),
)

s3_fmt.write_csv(BASE / "code" /"supplemental_table3.tsv", separator="\t")